# Planogram walkthrough
Runs the pipeline one step at a time on a single bay so you can see (and tune) each stage. Set the paths in the first cell. Needs `ANTHROPIC_API_KEY` in your environment and detector weights for steps 2 onwards.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

from planogram.config import Settings
from planogram import pipeline, export, render
from planogram.preprocess import load_image
from planogram.detect import detect_facings
from planogram.layout import shelf_lines
from planogram.identify import Identifier

PHOTO = Path("../data/photos/store01_bay03.jpg")
WEIGHTS = "../weights/sku110k_yolov8s.pt"
STORE, BAY, BAY_WIDTH_MM = "Store 01", "3", 1200
CORNERS = None   # e.g. [(120, 80), (2900, 60), (2950, 3900), (100, 3950)]  TL, TR, BR, BL
s = Settings()

## 1. Load and square up
EXIF rotation is fixed and the long edge capped at `s.max_side`. If the shelves look tilted, set `CORNERS`.

In [ ]:
img = load_image(PHOTO, s.max_side, CORNERS)
plt.figure(figsize=(10, 8)); plt.imshow(img); plt.axis("off");

## 2. Detect facings
One box per visible pack front. If boxes are missing on small items, raise `s.det_imgsz`; if price tickets are boxed, raise `s.det_conf`.

In [ ]:
boxes = detect_facings(img, WEIGHTS, s.det_conf, s.det_imgsz)
print(len(boxes), "facings")
fig, ax = plt.subplots(figsize=(10, 8)); ax.imshow(img); ax.axis("off")
for b in boxes:
    ax.add_patch(plt.Rectangle((b.x1, b.y1), b.w, b.h, fill=False, ec="#eb6834", lw=1))

## 3. Shelves and SKU runs
Boxes are grouped into shelves by their bottom edge, then neighbouring look-alike facings on a shelf merge into runs. `n` is how many Claude calls step 4 will make.
- Two shelves merged into one: lower `s.shelf_gap_frac`. One shelf split in two: raise it.
- One product split into several runs: lower `s.run_sim`. Different products merged: raise it, or use `embed_backend="clip"`.

In [ ]:
shelves, runs, n = pipeline.build_runs(img, boxes, s, embed_backend="colour")
print(f"{len(shelves)} shelves, {len(runs)} runs, {n} distinct-looking products")
pd.DataFrame([(r.shelf, r.position, r.facings, r.cluster) for r in runs],
             columns=["shelf", "position", "facings", "cluster"])

## 4. Identify with Claude (one call per cluster, cached)

In [ ]:
ident = Identifier(s.model, s.effort, "../.cache", s.workers)
leaders = {}
for r in sorted(runs, key=lambda r: -r.facings):
    leaders.setdefault(r.cluster, r)
crops = [pipeline.run_crop(img, leaders[k]) for k in range(n)]
ids = ident.identify_many(crops)
print(ident.usage)

fig, axes = plt.subplots(1, min(n, 8), figsize=(16, 3))
for ax, c, info in zip(axes if n > 1 else [axes], crops, ids):
    ax.imshow(c); ax.axis("off"); ax.set_title(f"{info['brand']}\n{info['variant']}", fontsize=8)

## 5. Planogram table and drawings

In [ ]:
runs = pipeline.merge_same_product(runs, key=lambda r: pipeline.sku_key(ids[r.cluster]))
meta = dict(store=STORE, bay=BAY, taken=export.photo_date(PHOTO),
            bay_px=img.shape[1] if CORNERS else None, bay_width_mm=BAY_WIDTH_MM)
df = export.from_runs(runs, ids, **meta)
out = Path("../outputs"); out.mkdir(exist_ok=True)
render.planogram_png(df, out / "walkthrough_planogram.png", f"{STORE} · Bay {BAY}")
display(df)
display(export.brand_summary(df))

## 6. Accuracy against a hand count
Stand at the bay and write down, per shelf, each product's brand, variant and facings in a CSV with columns `shelf, brand, variant, facings`. Then compare the pipeline against the one-shot LLM-only baseline. Do this for 3 to 5 bays before trusting the numbers.

In [ ]:
from planogram.evaluate import compare

truth = pd.read_csv("../data/hand_counts/store01_bay03.csv")
llm_only = export.from_bay(ident.identify_bay(img), img.shape[1], **meta)
pd.DataFrame({"pipeline": compare(df, truth), "llm-only": compare(llm_only, truth)})